# Chapitre 17 · Apprendre à obéir (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte « IA débranchée » vaut aussi pour les corrigés.

In [ ]:
# Mise en place (reprise de la leçon) : le minimum pour exécuter les validations.
import torch
import torch.nn as nn
import torch.nn.functional as F

# Le tokeniseur caractère et les trois tokens spéciaux de la leçon, en miniature.
SPECIAUX = {"<|user|>": "\x01", "<|assistant|>": "\x02", "<|fin|>": "\x03"}
mini_corpus = "Deux ? Trois."
chars = sorted(set(mini_corpus)) + sorted(SPECIAUX.values())
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
USR, ASS, FIN = stoi["\x01"], stoi["\x02"], stoi["\x03"]

### Exercice 1 · Les labels masqués — niveau ●

Dans la leçon, `fabriquer_flux_sft` a construit les labels à ta place. Refais le geste sur UNE paire : chaque token du prompt reçoit le label `-100`, chaque token de la réponse (et `<|fin|>`) garde son id.

In [ ]:
def exemple_sft(question, reponse):
    """Rend (ids, labels) pour une paire, deux listes alignées position par position."""
    prompt = [USR] + encode(question) + [ASS]        # la partie "consigne"
    cible = encode(reponse) + [FIN]                  # la partie "à apprendre"
    ids = prompt + cible
    labels = [-100] * len(prompt) + cible            # prompt masqué : -100
    return ids, labels

In [ ]:
# Validation : les labels masqués.
ids_t, labels_t = exemple_sft("Deux ?", "Trois.")
n_prompt = 1 + len("Deux ?") + 1                     # <|user|> + question + <|assistant|>
assert len(labels_t) == len(ids_t), "ids et labels doivent être alignés position par position"
assert labels_t[:n_prompt] == [-100] * n_prompt, "tout le prompt (tokens spéciaux inclus) doit être à -100"
assert labels_t[n_prompt:] == ids_t[n_prompt:], "la réponse (et <|fin|>) doit garder ses ids"
assert labels_t[-1] == FIN, "le dernier label doit être <|fin|> : s'arrêter fait partie de la leçon"
print("Labels OK : prompt à -100, réponse + <|fin|> conservés.")

### Exercice 2 · La loss qui ignore le prompt — niveau ●

La fonction `sft` de la leçon tient sur une ligne de loss. Réécris-la : une cross-entropy qui exclut toutes les positions dont le label vaut `-100`.

In [ ]:
def loss_sft(logits, labels):
    """Cross-entropy du SFT : les positions dont le label vaut -100 ne comptent pas."""
    return F.cross_entropy(logits.view(-1, vocab_size), labels.view(-1), ignore_index=-100)

In [ ]:
# Validation : la loss doit ignorer les positions à -100.
torch.manual_seed(1)
logits_t = torch.randn(2, 6, vocab_size)
labels_t = torch.randint(0, vocab_size, (2, 6))
labels_t[:, :3] = -100                               # la moitié des positions est masquée
garde = labels_t.view(-1) != -100
l_ref = F.cross_entropy(logits_t.view(-1, vocab_size)[garde], labels_t.view(-1)[garde])
l_exo = loss_sft(logits_t, labels_t)
assert torch.isclose(l_exo, l_ref, atol=1e-6), f"la loss doit valoir {l_ref.item():.6f}, pas {l_exo.item():.6f}"
print(f"Loss OK : {l_exo.item():.6f}, moyenne sur les seules positions non masquées.")

### Exercice 3 · Le forward de LoRA — niveau ●●

La classe ci-dessous reprend l'enveloppe de la leçon : W gelée, A en petite gaussienne, B à zéro. Il te manque le forward : la sortie de la couche gelée, plus la correction de rang faible à l'échelle alpha/r. Surveille tes shapes à chaque étape.

In [ ]:
class LoRALinearExo(nn.Module):
    """Enveloppe une couche nn.Linear : W est gelée, on apprend la correction (alpha/r) * B @ A."""

    def __init__(self, couche, r=8, alpha=16.0):
        super().__init__()
        self.couche = couche
        for p in self.couche.parameters():
            p.requires_grad = False                   # W ne bougera plus jamais
        self.echelle = alpha / r                      # le facteur alpha/r de la formule
        # A : (r, d_entree), petite gaussienne ; B : (d_sortie, r), zéros
        self.A = nn.Parameter(torch.randn(r, couche.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(couche.out_features, r))

    def forward(self, x):
        # x @ A.T : (B, T, d_entree) -> (B, T, r)   : on descend dans le goulot
        # ... @ B.T : (B, T, r) -> (B, T, d_sortie) : on remonte
        return self.couche(x) + (x @ self.A.T @ self.B.T) * self.echelle

In [ ]:
# Validation : le forward de LoRA.
torch.manual_seed(0)
lin_t = nn.Linear(96, 64, bias=False)                # une couche rectangulaire, exprès
lora_t = LoRALinearExo(lin_t, r=8, alpha=16.0)
x_t = torch.randn(2, 5, 96)
assert lora_t(x_t).shape == (2, 5, 64), f"shape obtenue : {tuple(lora_t(x_t).shape)}, attendue : (2, 5, 64)"
assert (lora_t(x_t) - lin_t(x_t)).abs().max().item() == 0.0, "avec B = 0, la sortie doit être EXACTEMENT celle de la couche gelée"
with torch.no_grad():
    lora_t.B.copy_(torch.randn_like(lora_t.B) * 0.1)  # on branche une correction non nulle
attendu = lin_t(x_t) + F.linear(x_t, lora_t.B @ lora_t.A) * lora_t.echelle   # Delta W = B @ A
ecart = (lora_t(x_t) - attendu).abs().max().item()
assert ecart < 1e-5, f"écart {ecart:.2e} : la correction doit valoir (alpha/r) * B(Ax)"
print("LoRA forward OK : correction nulle à l'init, formule exacte ensuite.")

### Exercice 4 · Fusionner l'adaptateur — niveau ●●●

Dernier geste du chapitre : absorber la correction dans W, une fois pour toutes, pour servir le modèle sans le détour de calcul LoRA. Ta fonction reçoit une couche LoRA de l'exercice 3 (la validation réutilise ta classe `LoRALinearExo`) et doit modifier `couche.weight` en place. L'ordre du produit compte.

In [ ]:
def fusionner(lora):
    """Absorbe la correction dans W : W <- W + (alpha/r) * B @ A (modifie lora.couche en place)."""
    with torch.no_grad():
        lora.couche.weight.data.add_((lora.B @ lora.A) * lora.echelle)

In [ ]:
# Validation : la fusion.
torch.manual_seed(0)
lin_t = nn.Linear(96, 96, bias=False)
lora_t = LoRALinearExo(lin_t, r=8, alpha=16.0)
with torch.no_grad():
    lora_t.B.copy_(torch.randn_like(lora_t.B) * 0.1)
x_t = torch.randn(2, 5, 96)
avant = lora_t(x_t)                                  # sortie avec adaptateur branché
fusionner(lora_t)
apres = lora_t.couche(x_t)                           # la couche seule, correction absorbée
ecart = (avant - apres).abs().max().item()
assert ecart < 1e-4, f"écart {ecart:.2e} : la couche fusionnée doit rendre la même sortie que LoRA"
print(f"Fusion OK : écart max {ecart:.2e}, la correction est absorbée dans W.")